# Embedding Models & Their Role in Retrieval — Code Companion

This notebook accompanies **Topic: Embedding Models & Their Role in Retrieval**.

We'll build a complete, working semantic search pipeline: turn text into embeddings,
measure similarity, chunk documents, and retrieve the most relevant chunk for a query —
the exact mechanism behind RAG.

**What you'll do:**
1. Implement cosine similarity from scratch and build intuition for it with simple
   hand-made vectors.
2. Generate real sentence embeddings with `sentence-transformers`.
3. Build a tiny semantic search engine — entirely in NumPy, no vector database needed.
4. Compare semantic search against plain keyword matching, side by side.
5. Implement and compare two chunking strategies.
6. Build a minimal two-stage retrieve-then-rerank pipeline.

Section 1 and 5 run anywhere with just NumPy. Sections 2-4 and 6 need
`pip install sentence-transformers` and internet access the first time (to download a
small embedding model).

## 1. Cosine Similarity, From Scratch

Before using any library, let's implement the formula directly and build intuition with
hand-crafted vectors, so the number itself is never a mystery.

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \, \|B\|}$$

In [1]:
import numpy as np

def cosine_similarity(a, b):
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)

# Hand-crafted 2D "embeddings" purely for intuition -- real ones have hundreds of dimensions
identical = (np.array([1.0, 0.0]), np.array([1.0, 0.0]))
similar   = (np.array([1.0, 0.0]), np.array([0.9, 0.2]))
unrelated = (np.array([1.0, 0.0]), np.array([0.0, 1.0]))
opposite  = (np.array([1.0, 0.0]), np.array([-1.0, 0.0]))

for label, (a, b) in [("Identical", identical), ("Similar", similar),
                        ("Unrelated (orthogonal)", unrelated), ("Opposite", opposite)]:
    print(f"{label:25s} cosine similarity = {cosine_similarity(a, b):+.3f}")

Identical                 cosine similarity = +1.000
Similar                   cosine similarity = +0.976
Unrelated (orthogonal)    cosine similarity = +0.000
Opposite                  cosine similarity = -1.000


This confirms the intuition from the slides directly: `1.0` for identical direction,
`~0.0` for unrelated (perpendicular) vectors, and `-1.0` for pointing in opposite
directions. Real embeddings live in hundreds of dimensions, but the formula and its
interpretation are identical.

## 2. Generating Real Sentence Embeddings

Now let's replace the hand-crafted vectors with real embeddings from a trained model.

> **Note:** needs `pip install sentence-transformers` and internet access the first time,
> to download the model.

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")   # small, fast, great for prototyping

sentences = [
    "The cat sat on the mat.",
    "A kitten rested on the rug.",
    "The stock market fell sharply today.",
    "Quarterly earnings disappointed investors.",
]

embeddings = model.encode(sentences)
print("Embedding shape:", embeddings.shape, "-> 4 sentences, 384 dimensions each")

print("\nPairwise similarities:")
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f"  [{i}] vs [{j}]: {sim:.3f}   \"{sentences[i][:30]}...\" vs \"{sentences[j][:30]}...\"")

W0812 13:36:27.422000 3684501 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0812 13:36:27.444000 3684501 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (4, 384) -> 4 sentences, 384 dimensions each

Pairwise similarities:
  [0] vs [1]: 0.612   "The cat sat on the mat...." vs "A kitten rested on the rug...."
  [0] vs [2]: 0.075   "The cat sat on the mat...." vs "The stock market fell sharply ..."
  [0] vs [3]: -0.002   "The cat sat on the mat...." vs "Quarterly earnings disappointe..."
  [1] vs [2]: 0.073   "A kitten rested on the rug...." vs "The stock market fell sharply ..."
  [1] vs [3]: 0.044   "A kitten rested on the rug...." vs "Quarterly earnings disappointe..."
  [2] vs [3]: 0.330   "The stock market fell sharply ..." vs "Quarterly earnings disappointe..."


The two sentences about cats should score noticeably higher against each other than
against either of the finance sentences, even though they don't share a single word in
common ("cat"/"kitten", "sat"/"rested", "mat"/"rug") — this is semantic search working
exactly as intended.

## 3. A Tiny Semantic Search Engine

A vector database (FAISS, Pinecone, Chroma, etc.) is optimized for searching millions of
vectors quickly — but the *core operation* it performs is simple enough to write
ourselves for a small collection. This is retrieval, stripped down to its essence.

In [3]:
documents = [
    "The Eiffel Tower is located in Paris, France.",
    "Python is a popular programming language for data science.",
    "The Great Wall of China stretches thousands of kilometers.",
    "Machine learning models are trained on large datasets.",
    "Mount Everest is the tallest mountain on Earth.",
    "Neural networks are inspired by the structure of the brain.",
]

doc_embeddings = model.encode(documents)

def semantic_search(query, doc_embeddings, documents, top_k=3):
    query_embedding = model.encode([query])[0]
    scores = [cosine_similarity(query_embedding, doc_emb) for doc_emb in doc_embeddings]
    ranked = sorted(zip(scores, documents), reverse=True)
    return ranked[:top_k]

query = "Tell me about famous landmarks"
results = semantic_search(query, doc_embeddings, documents, top_k=3)

print(f"Query: {query!r}\n")
for score, doc in results:
    print(f"  {score:.3f}  {doc}")

Query: 'Tell me about famous landmarks'

  0.325  Mount Everest is the tallest mountain on Earth.
  0.279  The Eiffel Tower is located in Paris, France.
  0.262  The Great Wall of China stretches thousands of kilometers.


Notice the top results are about the Eiffel Tower, the Great Wall, and Everest — none of
which contain the word "landmarks" — while the ML/Python documents correctly rank lower.
This exact pattern (embed documents once, embed the query, rank by cosine similarity) is
the retrieval half of RAG, and everything a vector database does at scale is optimizing
this same core search.

## 4. Semantic Search vs. Keyword Search, Side by Side

Let's make the gap concrete with the exact example from the slides: a query that
shares *no words at all* with the best matching document.

In [4]:
def keyword_search(query, documents, top_k=3):
    query_words = set(query.lower().split())
    scores = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words)
        scores.append(overlap)
    ranked = sorted(zip(scores, documents), reverse=True)
    return ranked[:top_k]

query = "affordable laptop"
docs = [
    "Budget-friendly notebook computers for students.",
    "Luxury sports cars for collectors.",
    "Cheap laptop deals available this week.",
    "High-end gaming desktop towers.",
]
doc_embs = model.encode(docs)

print(f"Query: {query!r}\n")

print("Keyword search results (word overlap count):")
for score, doc in keyword_search(query, docs):
    print(f"  {score}  {doc}")

print("\nSemantic search results (cosine similarity):")
for score, doc in semantic_search(query, doc_embs, docs, top_k=4):
    print(f"  {score:.3f}  {doc}")

Query: 'affordable laptop'

Keyword search results (word overlap count):
  1  Cheap laptop deals available this week.
  0  Luxury sports cars for collectors.
  0  High-end gaming desktop towers.

Semantic search results (cosine similarity):
  0.679  Cheap laptop deals available this week.
  0.661  Budget-friendly notebook computers for students.
  0.325  High-end gaming desktop towers.
  0.247  Luxury sports cars for collectors.


Keyword search completely misses "Budget-friendly notebook computers" (zero shared
words with "affordable laptop"), while semantic search ranks it near the top — this is
the exact blind spot from the slides, now demonstrated rather than just described.

## 5. Chunking Strategies, Implemented

Before documents can be embedded, long text needs to be split into chunks. Let's
implement and compare the two strategies from the slides on the same document.

In [5]:
long_document = (
    "Machine learning is a subset of artificial intelligence. "
    "It focuses on building systems that learn from data. "
    "Deep learning is a further subset of machine learning. "
    "It uses neural networks with many layers. "
    "Transformers are a type of deep learning architecture. "
    "They rely on a mechanism called self-attention. "
    "Self-attention lets models weigh the importance of different words. "
    "This has proven extremely effective for language tasks."
)

def fixed_size_chunk(text, chunk_size=60, overlap=0):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

def sentence_chunk(text, sentences_per_chunk=2):
    sentences = [s.strip() + "." for s in text.split(". ") if s.strip()]
    sentences[-1] = sentences[-1].rstrip(".") + "."  # avoid a doubled period on the last one
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunks.append(" ".join(sentences[i:i + sentences_per_chunk]))
    return chunks

print("FIXED-SIZE chunking (60 chars, no overlap) -- notice mid-sentence cuts:")
for i, c in enumerate(fixed_size_chunk(long_document)):
    print(f"  [{i}] {c!r}")

print("\nSENTENCE-based chunking (2 sentences per chunk) -- respects sentence boundaries:")
for i, c in enumerate(sentence_chunk(long_document)):
    print(f"  [{i}] {c!r}")

FIXED-SIZE chunking (60 chars, no overlap) -- notice mid-sentence cuts:
  [0] 'Machine learning is a subset of artificial intelligence. It '
  [1] 'focuses on building systems that learn from data. Deep learn'
  [2] 'ing is a further subset of machine learning. It uses neural '
  [3] 'networks with many layers. Transformers are a type of deep l'
  [4] 'earning architecture. They rely on a mechanism called self-a'
  [5] 'ttention. Self-attention lets models weigh the importance of'
  [6] ' different words. This has proven extremely effective for la'
  [7] 'nguage tasks.'

SENTENCE-based chunking (2 sentences per chunk) -- respects sentence boundaries:
  [0] 'Machine learning is a subset of artificial intelligence. It focuses on building systems that learn from data.'
  [1] 'Deep learning is a further subset of machine learning. It uses neural networks with many layers.'
  [2] 'Transformers are a type of deep learning architecture. They rely on a mechanism called self-attention.'
  [3] '

Fixed-size chunking cuts sentences awkwardly wherever the character count happens to
land; sentence-based chunking always keeps complete ideas together. Neither is
universally "correct" — the tradeoff from the slides (simplicity and predictable size vs.
respecting natural boundaries) is now visible directly in the output.

## 6. Retrieve, Then Rerank

A minimal two-stage pipeline: fast embedding search narrows a large candidate pool, then
a slower but more accurate cross-encoder reranks just the top few.

> **Note:** the reranking step below needs `pip install sentence-transformers` (already
> installed above) — `CrossEncoder` downloads a small additional model on first use.

In [6]:
from sentence_transformers import CrossEncoder

candidates = [
    "The Eiffel Tower was completed in 1889 in Paris.",
    "Paris is the capital city of France.",
    "The Louvre Museum is also located in Paris.",
    "London is the capital of the United Kingdom.",
    "The Statue of Liberty is in New York Harbor.",
]
candidate_embeddings = model.encode(candidates)

query = "When was the Eiffel Tower built?"

# Stage 1: fast retrieval with embeddings -- narrow a big pool down to a handful
top_k_results = semantic_search(query, candidate_embeddings, candidates, top_k=3)
print("Stage 1 -- embedding retrieval (fast, approximate):")
for score, doc in top_k_results:
    print(f"  {score:.3f}  {doc}")

# Stage 2: rerank just those candidates with a slower, more accurate cross-encoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
pairs = [(query, doc) for _, doc in top_k_results]
rerank_scores = reranker.predict(pairs)

reranked = sorted(zip(rerank_scores, [doc for _, doc in top_k_results]), reverse=True)
print("\nStage 2 -- cross-encoder reranking (slower, more accurate):")
for score, doc in reranked:
    print(f"  {score:.3f}  {doc}")

Stage 1 -- embedding retrieval (fast, approximate):
  0.813  The Eiffel Tower was completed in 1889 in Paris.
  0.329  The Louvre Museum is also located in Paris.
  0.269  Paris is the capital city of France.


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.67k [00:00<?, ?B/s]


Stage 2 -- cross-encoder reranking (slower, more accurate):
  10.016  The Eiffel Tower was completed in 1889 in Paris.
  -10.720  Paris is the capital city of France.
  -10.843  The Louvre Museum is also located in Paris.


The reranker looks at the query and each candidate *together* (rather than comparing two
independently-computed embeddings), which is more accurate but too slow to run against
every document in a large collection — exactly why it's used only on the shortlist that
embedding search already narrowed down.

## Recap & Try It Yourself

You just built a complete, working retrieval pipeline:
- Cosine similarity, implemented from scratch and sanity-checked on hand-crafted vectors.
- Real sentence embeddings, and pairwise similarity between them.
- A working semantic search engine over a small document collection.
- A direct, concrete comparison against keyword search on a query with zero word overlap.
- Two chunking strategies, and what each one trades off.
- A two-stage retrieve-then-rerank pipeline.

**Things to try:**
1. In Section 3, add 5-10 more documents on entirely different topics and try queries
   that use synonyms rather than exact document words.
2. In Section 5, add an overlapping fixed-size chunker (`overlap > 0`) and see how it
   changes which sentences get split across chunk boundaries.
3. In Section 6, swap the query for one where Stage 1 and Stage 2 disagree on the top
   result, and think through why the reranker might have changed the ordering.
4. Try a much larger `all-mpnet-base-v2` embedding model instead of `all-MiniLM-L6-v2`
   and compare similarity scores and retrieval quality.